In [31]:
import os
import asyncio
from IPython.display import display,Markdown

In [32]:
import resend
from agents import Agent,Runner,function_tool
from dotenv import load_dotenv,find_dotenv
from openai import AsyncOpenAI,OpenAI
from agents import set_default_openai_client,OpenAIChatCompletionsModel,trace,set_tracing_export_api_key

In [33]:
load_dotenv(find_dotenv(),override=True)

True

In [34]:
resend.api_key = os.getenv("RESEND_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL")

In [35]:
tracing_api_key = os.getenv("OPENAI_API_KEY")
set_tracing_export_api_key(tracing_api_key)

In [36]:
client_gemini = AsyncOpenAI(base_url=gemini_base_url, api_key=gemini_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite",openai_client=client_gemini)

In [37]:
instructions1 = "You are a sales agent working for CoolAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails. Return Only TEXT MAIL NOTHING ELSE"

instructions2 = "You are a humorous, engaging sales agent working for CoolAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response. Return Only TEXT MAIL NOTHING ELSE"

instructions3 = "You are a busy sales agent working for CoolAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails. Return Only TEXT MAIL NOTHING ELSE"

In [38]:
sales_agent1 = Agent(name="Professional Sales Agent",instructions=instructions1,model=gemini_model)
sales_agent2 = Agent(name="Engaging Sales Agent",instructions=instructions2,model=gemini_model)
sales_agent3 = Agent(name="Busy Sales Agent",instructions=instructions3,model=gemini_model)

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

In [ ]:
for output in outputs:
    print(output + "\n\n")

In [39]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    params: resend.Emails.SendParams = {
    "from": "Acme <onboarding@resend.dev>",
    "to": ["jayanthreddykonda1@gmail.com"],
    "subject": "Sales",
    "html": f"{body}",
    }
    resend.Emails.send(params)
    return {"status": "success"}

In [40]:
description = "Write a cold sales email"

Agent1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
Agent2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
Agent3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [41]:
tools = [Agent1, Agent2, Agent3, send_email]

In [42]:
instructions = """
You are a Sales Manager at CoolAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=gemini_model)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sale manager"):
    result = await Runner.run(sales_manager, message)

In [43]:
print(result.final_output)

The best email (Draft 3) has been sent to the prospects.


## Handoff

In [44]:
@function_tool
def send_email_with_subject(subject: str,body: str):
    """
    Send out an email with the given body to all sales prospects

    Args:
        subject: Subject to be sent in mail.
        body: body to the sent as html in mail.
    """
    params: resend.Emails.SendParams = {
    "from": "Acme <onboarding@resend.dev>",
    "to": ["jayanthreddykonda1@gmail.com"],
    "subject": f"{subject}",
    "html": f"{body}",
    }
    resend.Emails.send(params)
    return {"status": "success"}

In [45]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=gemini_model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=gemini_model)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [46]:
tools_for_emailer = [subject_tool, html_tool, send_email_with_subject]

In [47]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools_for_emailer,
    model=gemini_model,
    handoff_description="Convert an email to HTML and send it")

In [48]:
tools_for_sales_manager = [Agent1, Agent2, Agent3]
handoffs = [emailer_agent]

In [49]:
sales_manager_instructions = """
You are a Sales Manager at CoolAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager_auto = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools_for_sales_manager,
    handoffs=handoffs,
    model=gemini_model)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager_auto, message)

In [50]:
result.final_output

'The cold sales email has been successfully sent with the subject line "Tired of overpaying for SOC2?".'